# ***AI-Driven Self-Serve Analytical Intelligence Platform***

This notebook implements a prototype of an **AI-powered autonomous analytics system** that converts natural language business questions into **SQL queries, insights, and visualizations**.

---

### Objective

Enable business users to:
- Ask questions in natural language
- Automatically retrieve relevant data
- Generate insights with explanations
- Visualize results instantly

---

### Key Capabilities

- Natural Language Understanding (LLM-based)
- NL → SQL generation with guardrails
- Hypothesis-based analytical reasoning
- Automatic chart generation
- Schema-aware retrieval (RAG)
- Confidence scoring

---

### Architecture Flow
User Query → Query Understanding → RAG (Schema Retrieval)
→ Hypothesis Engine → NL2SQL → SQL Execution → Answer + Chart + Confidence


---

### Current Scope

This notebook is:
- **A near-production prototype**
- Modularized into logical blocks (convertible to `.py`)
- Focused on **Payments / Transactions dataset**

---

### Future Enhancements

- Multi-hypothesis evaluation engine
- Governance layer (RBAC, PII masking)
- Feedback loop (RLHF)
- API deployment (FastAPI)
- Dashboard UI (Streamlit / React)

---

## ***1. Setup***
**Purpose:** Install libraries and initialize environment  
**Next:** Move to config/env-based setup  
**Status:** Final

In [17]:


!pip install pandas numpy faker duckdb openai faiss-cpu sentence-transformers vega_datasets altair

import pandas as pd
import numpy as np
import duckdb
import random
from faker import Faker
from datetime import datetime, timedelta
import json
import altair as alt

# Embeddings
from sentence_transformers import SentenceTransformer
import faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.7 MB/s eta 0:00:00


Getting API Key Securely


In [18]:
# Secure API Key Input
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

# OpenAI Client
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

Enter your OpenAI API Key: ··········


## ***2.Dataset Generation***

In [19]:

fake = Faker()

def generate_payments_data(n=10000):
    data = []

    for _ in range(n):
        data.append([
            fake.uuid4(),
            random.randint(1000, 1100),
            round(random.uniform(10, 1000), 2),
            random.choice(["SUCCESS", "FAILED", "PENDING"]),
            random.choice(["CARD", "UPI", "WALLET"]),
            random.choice(["India", "USA", "UK"]),
            fake.date_time_between(start_date='-90d', end_date='now')
        ])

    df = pd.DataFrame(data, columns=[
        "txn_id", "user_id", "amount", "status",
        "payment_method", "country", "created_at"
    ])

    df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_localize(None)

    return df

df = generate_payments_data(15000)

In [20]:
df.head()

,txn_id,user_id,amount,status,payment_method,country,created_at
0,42cc6918-4237-406b-bd70-d29c42ccfa8a,1060,750.61,PENDING,WALLET,USA,2026-03-03 22:56:45.983845
1,11ce55bd-8283-4d3c-9fdc-34d24387e646,1062,224.96,SUCCESS,UPI,UK,2026-02-06 16:31:05.558084
2,9ceadce7-827c-48a6-930e-480cba90fb02,1081,751.70,SUCCESS,WALLET,India,2026-02-20 15:32:58.905090
3,6ea99f22-c334-4a52-9b66-593565eb4fcb,1073,607.71,PENDING,CARD,USA,2026-04-12 12:47:13.905358
4,7042734b-a0bb-4ee4-ae59-b77be45921f8,1029,537.82,PENDING,CARD,USA,2026-03-16 16:12:22.675189


## ***3. Schema Retrieval (RAG)***
**Purpose:** Retrieve relevant schema using embeddings  
**Next:** Replace FAISS with pgvector  
**Status:** Partial

In [21]:

schema_docs = [
    "txn_id: unique transaction id",
    "user_id: user identifier",
    "amount: transaction amount",
    "status: SUCCESS/FAILED/PENDING",
    "payment_method: CARD/UPI/WALLET",
    "country: transaction country",
    "created_at: timestamp of transaction"
]

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(schema_docs)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

def retrieve_schema(query, k=3):
    q_emb = model.encode([query])
    distances, indices = index.search(np.array(q_emb), k)
    return [schema_docs[i] for i in indices[0]]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## ***4. Query Understanding***

**Purpose:** Extract intent, metric, and time filters  
**Next:** Improve accuracy + structured outputs  
**Status:** Partial

In [34]:
def understand_query(query):

    prompt = f"""
    You are a strict JSON generator.

    Extract:
    - complexity: "simple" or "complex"
    - metric: (revenue, transactions, etc.)
    - time_filter: (7d, 30d, null)

    Return ONLY valid JSON. No explanation. No text.

    Example:
    {{
      "complexity": "simple",
      "metric": "revenue",
      "time_filter": null
    }}

    Query: {query}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    raw = response.choices[0].message.content.strip()
    print("LLM RAW OUTPUT:", raw)

    return safe_json_load(raw)

Helper Function to Load JSON safely

In [35]:
import re
import json

def safe_json_load(text):
    try:
        return json.loads(text)
    except:
        # Try extracting JSON block
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        else:
            raise ValueError("Failed to parse JSON from LLM output:\n" + text)

## ***5. Hypothesis Engine***

**Purpose:** Generate analytical hypotheses  
**Next:** Add hypothesis evaluation  
**Status:** Partial

In [23]:
def generate_hypotheses(query, complexity):

    if complexity == "simple":
        return ["Direct computation"]

    prompt = f"""
    Generate 3 analytical hypotheses for:
    {query}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.choices[0].message.content
    hypotheses = [h.strip("- ").strip() for h in raw.split("\n") if h.strip()]
    return hypotheses

## ***6. NL → SQL (NL2SQL Engine)***

**Purpose:** Convert query to SQL  
**Next:** Add stronger validation + optimization  
**Status:** Partial

In [37]:
def generate_sql(query, schema_context):

    prompt = f"""
    You are a SQL generator.

    Convert the question into a valid DuckDB SQL query.

    STRICT RULES:
    - Output ONLY SQL
    - No explanation
    - No markdown (no ```sql)
    - No text before or after
    - Table name: df
    - Use column: created_at for time
    - Use status = 'SUCCESS' for revenue
    - Do NOT assume columns not in schema

    Schema:
    {schema_context}

    Example:
    SELECT SUM(amount) FROM df WHERE status = 'SUCCESS';

    Query: {query}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.choices[0].message.content.strip()

    return clean_sql(raw)

Helper to fix time filter issues with the NOW()

In [57]:
def fix_time_filters(sql):

    import re
    from datetime import datetime, timedelta

    # Match ANY number of days
    match = re.search(r"NOW\(\)\s*-\s*INTERVAL\s*'(\d+)\s*days'", sql, re.IGNORECASE)

    if match:
        days = int(match.group(1))
        cutoff = datetime.now() - timedelta(days=days)
        cutoff_str = cutoff.strftime("%Y-%m-%d %H:%M:%S")

        sql = re.sub(
            r"NOW\(\)\s*-\s*INTERVAL\s*'\d+\s*days'",
            f"TIMESTAMP '{cutoff_str}'",
            sql,
            flags=re.IGNORECASE
        )

    # Fix CURRENT_DATE
    sql = re.sub(
        r"CURRENT_DATE",
        f"TIMESTAMP '{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}'",
        sql,
        flags=re.IGNORECASE
    )

    return sql

Function to clean the Raw SQL produced by LLM

In [38]:
import re

def clean_sql(text):
    """
    Extracts only SQL from LLM output
    """

    # Remove markdown blocks
    text = re.sub(r"```sql|```", "", text, flags=re.IGNORECASE)

    # Extract SELECT statement
    match = re.search(r"(SELECT .*?;)", text, re.IGNORECASE | re.DOTALL)

    if match:
        return match.group(1).strip()

    # fallback: find SELECT till end
    match = re.search(r"(SELECT .* )", text, re.IGNORECASE | re.DOTALL)

    if match:
        return match.group(1).strip()

    raise ValueError("No valid SQL found in LLM output:\n" + text)

Guardrail for SQL Query

In [25]:
# SQL VALIDATION (GUARDRAIL)

def validate_sql(sql):
    forbidden = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER"]

    for word in forbidden:
        if word in sql.upper():
            raise ValueError(f"Unsafe SQL detected: {word}")

    return True

## ***7. SQL Execution***

**Purpose:** Run queries using DuckDB  
**Next:** Add cost + performance controls  
**Status:** Final

In [26]:
def execute_sql(query, df):
    con = duckdb.connect()
    con.register("df", df)

    try:
        result = con.execute(query).fetchdf()
        return result, True
    except Exception as e:
        print("SQL Error:", e)
        return pd.DataFrame(), False

## ***8. Chart Generation***

**Purpose:** Auto-generate charts from results  
**Next:** Smarter chart selection  
**Status:** Partial

In [59]:
def generate_chart(df_result):

    if df_result is None or df_result.empty:
        return None

    df_plot = df_result.copy()

    # FIX: Rename columns to clean names
    df_plot.columns = [str(col).lower().replace("(", "").replace(")", "").replace(" ", "_")
                       for col in df_plot.columns]

    cols = df_plot.columns.tolist()

    # Case 1: need at least 2 columns
    if len(cols) < 2:
        return None

    x_col, y_col = cols[0], cols[1]

    # Case 2: time series
    if "date" in x_col or "day" in x_col:
        chart = alt.Chart(df_plot).mark_line().encode(
            x=alt.X(x_col, type='temporal'),
            y=alt.Y(y_col, type='quantitative')
        ).properties(width=500, height=300)
        return chart

    # Case 3: categorical bar
    chart = alt.Chart(df_plot).mark_bar().encode(
        x=alt.X(x_col, type='nominal'),
        y=alt.Y(y_col, type='quantitative')
    ).properties(width=500, height=300)

    return chart

## ***9. Confidence Scoring***

**Purpose:** Estimate reliability of results  
**Next:** Add data quality signals  
**Status:** Partial

In [28]:
# 9. CONFIDENCE SCORING

def compute_confidence(sql_success, row_count, hypothesis_count):

    score = 0

    if sql_success:
        score += 0.4

    if row_count > 0:
        score += 0.3

    if hypothesis_count > 1:
        score += 0.2

    if row_count > 10:
        score += 0.1

    return round(min(score, 1.0), 2)

## ***10. Answer Synthesis***

**Purpose:** Generate business-friendly answers  
**Next:** Add reasoning + explanation layer  
**Status:** Partial

In [50]:
# 10. ANSWER SYNTHESIS

def synthesize_answer(query, result_df):

    # 1. HARD SAFETY (RULE-BASED FIRST)
    if result_df is not None and not result_df.empty:

        # Single value result
        if result_df.shape[1] == 1:
            val = result_df.iloc[0, 0]

            if "revenue" in query.lower():
                return f"The total revenue is ${val:,.2f}."

            if "transaction" in query.lower():
                return f"The total number of transactions is {int(val):,}."

            return f"The result is {val}."

    # 2. LLM FALLBACK (STRICT CONTROL)
    prompt = f"""
    You are a data analyst.

    Generate a concise business answer ONLY based on the result.

    STRICT RULES:
    - Do NOT assume anything beyond the data
    - Do NOT mention revenue drop unless explicitly asked
    - Do NOT reuse previous answers
    - Keep answer specific to the query
    - 1–2 lines max

    Query: {query}

    Result:
    {result_df.to_string()}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

## ***11. Pipeline***

**Purpose:** End-to-end orchestration  
**Next:** Convert to modular services  
**Status:** Final

In [43]:
def display_output(output):

    print("\n" + "="*60)
    print("BUSINESS INSIGHT")
    print("="*60)
    print(output["answer"])

    print("\n SQL USED")
    print(output["sql"])

    print("\n HYPOTHESES")
    for h in output["hypotheses"]:
        print(f"- {h}")

    print(f"\n CONFIDENCE: {output['confidence']}")

    if output["chart"]:
        display(output["chart"])

In [53]:
def run_pipeline(query, df):

    print(" Query:", query)

    # RAG
    schema_context = retrieve_schema(query)

    # Understanding
    parsed = understand_query(query)
    complexity = parsed.get("complexity", "simple")

    # FIX: force complex for "why" queries
    if "why" in query.lower():
      complexity = "complex"

    # Hypotheses
    hypotheses = generate_hypotheses(query, complexity)

    # SQL (clean + fix)
    raw_sql = generate_sql(query, schema_context)

    try:
        sql = clean_sql(raw_sql)
        sql = fix_time_filters(sql)
    except:
        sql = raw_sql
    print(" RAW SQL:", raw_sql)
    print(" FIXED SQL:", sql)

    # Execute
    try:
        validate_sql(sql)
        result, sql_success = execute_sql(sql, df)
    except:
        result = pd.DataFrame()
        sql_success = False

    # Chart
    chart = generate_chart(result)

    # Answer
    if sql_success and not result.empty:
        answer = synthesize_answer(query, result)
    else:
        answer = "Could not retrieve reliable results."

    # Confidence
    confidence = compute_confidence(
        sql_success,
        len(result),
        len(hypotheses)
    )

    output = {
        "answer": answer,
        "sql": sql,
        "hypotheses": hypotheses,
        "confidence": confidence,
        "chart": chart
    }

    # USE DISPLAY FUNCTION HERE
    display_output(output)

    return output

## ***Below are a few test Queries:***


------


In [55]:
queries = [
    "What is total revenue?"
]

for q in queries:
    run_pipeline(q, df)


 Query: What is total revenue?
LLM RAW OUTPUT: {
  "complexity": "simple",
  "metric": "revenue",
  "time_filter": null
}
 RAW SQL: SELECT SUM(amount) FROM df WHERE status = 'SUCCESS';
 FIXED SQL: SELECT SUM(amount) FROM df WHERE status = 'SUCCESS';

BUSINESS INSIGHT
The total revenue is $2,495,827.60.

 SQL USED
SELECT SUM(amount) FROM df WHERE status = 'SUCCESS';

 HYPOTHESES
- Direct computation

 CONFIDENCE: 0.7


In [61]:
query = "Show number of transactions by payment method"
run_pipeline(query, df)

 Query: Show number of transactions by payment method
LLM RAW OUTPUT: {
  "complexity": "simple",
  "metric": "transactions",
  "time_filter": null
}
 RAW SQL: SELECT payment_method, COUNT(txn_id) FROM df WHERE status = 'SUCCESS' GROUP BY payment_method;
 FIXED SQL: SELECT payment_method, COUNT(txn_id) FROM df WHERE status = 'SUCCESS' GROUP BY payment_method;

BUSINESS INSIGHT
The number of transactions by payment method is as follows: CARD - 1668, UPI - 1645, WALLET - 1669.

 SQL USED
SELECT payment_method, COUNT(txn_id) FROM df WHERE status = 'SUCCESS' GROUP BY payment_method;

 HYPOTHESES
- Direct computation

 CONFIDENCE: 0.7


alt.Chart(...)

{'answer': 'The number of transactions by payment method is as follows: CARD - 1668, UPI - 1645, WALLET - 1669.',
 'sql': "SELECT payment_method, COUNT(txn_id) FROM df WHERE status = 'SUCCESS' GROUP BY payment_method;",
 'hypotheses': ['Direct computation'],
 'confidence': 0.7,
 'chart': alt.Chart(...)}

In [62]:
q = "Show total revenue by country"
run_pipeline(q,df)

 Query: Show total revenue by country
LLM RAW OUTPUT: {
  "complexity": "simple",
  "metric": "revenue",
  "time_filter": null
}
 RAW SQL: SELECT country, SUM(amount) FROM df WHERE status = 'SUCCESS' GROUP BY country;
 FIXED SQL: SELECT country, SUM(amount) FROM df WHERE status = 'SUCCESS' GROUP BY country;

BUSINESS INSIGHT
The total revenue by country is as follows: India - 841,197.13, USA - 809,277.41, UK - 845,353.06.

 SQL USED
SELECT country, SUM(amount) FROM df WHERE status = 'SUCCESS' GROUP BY country;

 HYPOTHESES
- Direct computation

 CONFIDENCE: 0.7


alt.Chart(...)

{'answer': 'The total revenue by country is as follows: India - 841,197.13, USA - 809,277.41, UK - 845,353.06.',
 'sql': "SELECT country, SUM(amount) FROM df WHERE status = 'SUCCESS' GROUP BY country;",
 'hypotheses': ['Direct computation'],
 'confidence': 0.7,
 'chart': alt.Chart(...)}

In [60]:
query = "Show daily revenue for the last 30 days"
run_pipeline(query, df)

 Query: Show daily revenue for the last 30 days
LLM RAW OUTPUT: {
  "complexity": "simple",
  "metric": "revenue",
  "time_filter": "30d"
}
 RAW SQL: SELECT DATE(created_at) AS day, SUM(amount) FROM df WHERE status = 'SUCCESS' AND created_at >= NOW() - INTERVAL '30 days' GROUP BY day;
 FIXED SQL: SELECT DATE(created_at) AS day, SUM(amount) FROM df WHERE status = 'SUCCESS' AND created_at >= TIMESTAMP '2026-03-17 19:35:09' GROUP BY day;

BUSINESS INSIGHT
Daily revenue for the last 30 days varies, with the highest recorded at $35,410.51 on March 19, 2026, and the lowest at $6,048.78 on March 17, 2026.

 SQL USED
SELECT DATE(created_at) AS day, SUM(amount) FROM df WHERE status = 'SUCCESS' AND created_at >= TIMESTAMP '2026-03-17 19:35:09' GROUP BY day;

 HYPOTHESES
- Direct computation

 CONFIDENCE: 0.8


alt.Chart(...)

{'answer': 'Daily revenue for the last 30 days varies, with the highest recorded at $35,410.51 on March 19, 2026, and the lowest at $6,048.78 on March 17, 2026.',
 'sql': "SELECT DATE(created_at) AS day, SUM(amount) FROM df WHERE status = 'SUCCESS' AND created_at >= TIMESTAMP '2026-03-17 19:35:09' GROUP BY day;",
 'hypotheses': ['Direct computation'],
 'confidence': 0.8,
 'chart': alt.Chart(...)}

In [54]:


queries = [
    "What is total revenue?",
    "How many transactions last week?",
    "Why did revenue drop last week?"
]

for q in queries:
    run_pipeline(q, df)


 Query: What is total revenue?
LLM RAW OUTPUT: {
  "complexity": "simple",
  "metric": "revenue",
  "time_filter": null
}
 RAW SQL: SELECT SUM(amount) FROM df WHERE status = 'SUCCESS';
 FIXED SQL: SELECT SUM(amount) FROM df WHERE status = 'SUCCESS';

BUSINESS INSIGHT
The total revenue is $2,495,827.60.

 SQL USED
SELECT SUM(amount) FROM df WHERE status = 'SUCCESS';

 HYPOTHESES
- Direct computation

 CONFIDENCE: 0.7
 Query: How many transactions last week?
LLM RAW OUTPUT: {
  "complexity": "simple",
  "metric": "transactions",
  "time_filter": "7d"
}
 RAW SQL: SELECT COUNT(txn_id) FROM df WHERE created_at >= NOW() - INTERVAL '7 days';
 FIXED SQL: SELECT COUNT(txn_id) FROM df WHERE created_at >= TIMESTAMP '2026-04-09 19:30:30';

BUSINESS INSIGHT
The total number of transactions is 1,183.

 SQL USED
SELECT COUNT(txn_id) FROM df WHERE created_at >= TIMESTAMP '2026-04-09 19:30:30';

 HYPOTHESES
- Direct computation

 CONFIDENCE: 0.7
 Query: Why did revenue drop last week?
LLM RAW OUTPUT: {